# YOLOv11n Baseline — Single Dataset

- 모델: YOLOv11n (COCO pretrained)
- 데이터: AI Hub single 경구약제 (1-class: pill)
- 하이퍼파라미터: AdamW lr=0.002, 30 epochs, 640px, bs=16

In [1]:
!pip install ultralytics -q
from google.colab import auth
auth.authenticate_user()

import pandas as pd
import numpy as np
from pathlib import Path
import yaml, os, shutil, zipfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 24.9 MB/s eta 0:00:00


## 1. Manifest 로드 및 스키마 확인

In [2]:
MANIFEST_PATH = "gs://pilliot-raw-data-2026/04_manifests/detection_single.csv"
df = pd.read_csv(MANIFEST_PATH)

print("columns:", df.columns.tolist())
print(df.head(2).to_string())
print("\n--- zip별 이미지 수 ---")
print(df.groupby(['split_type', 'label_zip_name']).size().reset_index(name='count').to_string())

columns: ['dataset_type', 'split_type', 'label_zip_name', 'image_file', 'item_seq', 'dl_name', 'width', 'height', 'category_id', 'bbox_x', 'bbox_y', 'bbox_w', 'bbox_h', 'area']
  dataset_type split_type label_zip_name                       image_file   item_seq        dl_name  width  height  category_id  bbox_x  bbox_y  bbox_w  bbox_h     area
0       single      train   TL_21_단일.zip  K-000027_0_2_0_0_75_000_200.png  195900043  아네모정 90mg/PTP    976    1280            1   294.0   475.0   285.0   279.0  79515.0
1       single      train   TL_21_단일.zip  K-000027_0_2_0_0_75_020_200.png  195900043  아네모정 90mg/PTP    976    1280            1   303.0   461.0   275.0   276.0  75900.0

--- zip별 이미지 수 ---
    split_type label_zip_name  count
0        train   TL_10_단일.zip  64800
1        train   TL_11_단일.zip  64800
2        train   TL_12_단일.zip  64584
3        train   TL_13_단일.zip  64800
4        train   TL_14_단일.zip  64800
5        train   TL_15_단일.zip  64800
6        train   TL_16_단일.zip  64584


## 2. Subset 선택
- AI Hub 기본 split 승계 (split_type 기준)
- train: TS_1_단일, TS_2_단일 / val: VS_1_단일

In [3]:
TRAIN_LABEL_ZIPS = ['TL_38_단일.zip']
VAL_LABEL_ZIPS   = ['VL_1_단일.zip']

def label_to_image_zip(lzip):
    return lzip.replace('TL_', 'TS_').replace('VL_', 'VS_')

train_df = df[df['label_zip_name'].isin(TRAIN_LABEL_ZIPS)].copy()
val_df   = df[df['label_zip_name'].isin(VAL_LABEL_ZIPS)].copy()

print(f"train: {len(train_df):,}  val: {len(val_df):,}")
print("\n--- train item_seq 분포 (상위 20) ---")
print(train_df['item_seq'].value_counts().head(20))

train: 12,852  val: 14,472

--- train item_seq 분포 (상위 20) ---
item_seq
200300269    324
200300251    324
200300431    324
200300412    324
200300312    324
200300463    324
200300418    324
200300393    324
200300408    324
200300953    324
200300808    324
200300840    324
200300913    324
200300769    324
200300657    324
200300598    324
200300579    324
200300496    324
200301033    324
200300470    216
Name: count, dtype: int64


## 3. 이미지 다운로드
zip 다운로드 → 압축 해제 → zip 삭제 (디스크 공간 재활용)

In [4]:
!df -h /content

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   43G   70G  39% /


In [5]:
import shutil
shutil.rmtree('/content/images', ignore_errors=True)

In [6]:
GCS_BASE = "gs://pilliot-raw-data-2026/00_original_zip/aihub_576_oral_pill/single"

def download_and_extract(label_zips, split, subset_df):
    dst = Path(f"/content/images/{split}")
    dst.mkdir(parents=True, exist_ok=True)
    for lzip in label_zips:
        img_zip = label_to_image_zip(lzip)
        gcs_src = f"{GCS_BASE}/{split}/images/{img_zip}"
        print(f"Downloading {img_zip} ...")
        ret = os.system(f'gcloud storage cp "{gcs_src}" /content/tmp.zip')
        if ret != 0 or not Path('/content/tmp.zip').exists():
            print(f"  ERROR: download failed for {img_zip}")
            continue

        selected_set = set(subset_df[subset_df['label_zip_name'] == lzip]['image_file'].tolist())

        with zipfile.ZipFile('/content/tmp.zip') as zf:
            for member in zf.namelist():
                fname = Path(member).name
                if fname in selected_set:
                    data = zf.read(member)
                    (dst / fname).write_bytes(data)

        os.remove('/content/tmp.zip')
        used = shutil.disk_usage('/content').used / 1e9
        n = len([p for p in dst.glob('*') if p.suffix in {'.jpg', '.png'}])
        print(f"  done. images: {n:,}  disk used: {used:.1f} GB")

download_and_extract(TRAIN_LABEL_ZIPS, 'train',      train_df)
download_and_extract(VAL_LABEL_ZIPS,   'validation', val_df)

  done. images: 12,852  disk used: 65.5 GB
  done. images: 14,472  disk used: 86.0 GB


## 4. YOLO 라벨 변환
COCO `[x, y, w, h]` (픽셀) → YOLO `[xc, yc, w, h]` (정규화)

In [7]:
def coco_to_yolo(row):
    bx, by, bw, bh = row['bbox_x'], row['bbox_y'], row['bbox_w'], row['bbox_h']
    iw, ih = row['width'], row['height']
    if pd.isna(bx) or bw < 1 or bh < 1:
        return None
    xc = (bx + bw / 2) / iw
    yc = (by + bh / 2) / ih
    return f"0 {xc:.6f} {yc:.6f} {bw/iw:.6f} {bh/ih:.6f}"

def write_labels(subset_df, split):
    label_dir = Path(f"/content/labels/{split}")
    label_dir.mkdir(parents=True, exist_ok=True)
    skipped = 0
    for fname, grp in subset_df.groupby('image_file'):
        lines = []
        for _, row in grp.iterrows():
            line = coco_to_yolo(row)
            if line:
                lines.append(line)
            else:
                skipped += 1
        if lines:
            (label_dir / (Path(fname).stem + '.txt')).write_text('\n'.join(lines))
    n_files = len(list(label_dir.glob('*.txt')))
    print(f"{split}: {n_files} 라벨 파일 생성, {skipped} annotation 스킵")

write_labels(train_df, 'train')
write_labels(val_df,   'validation')

train: 12852 라벨 파일 생성, 0 annotation 스킵
validation: 14472 라벨 파일 생성, 0 annotation 스킵


## 5. 이미지-라벨 매칭 검증

In [8]:
def validate_pairs(split):
    imgs   = {p.stem for p in Path(f"/content/images/{split}").glob("*") if p.suffix in {'.jpg', '.png'}}
    labels = {p.stem for p in Path(f"/content/labels/{split}").glob("*.txt")}
    no_label = imgs - labels
    no_image = labels - imgs
    print(f"[{split}] images: {len(imgs):,}, labels: {len(labels):,}")
    print(f"  라벨 없는 이미지: {len(no_label)}")
    print(f"  이미지 없는 라벨: {len(no_image)}")

validate_pairs('train')
validate_pairs('validation')

[train] images: 12,852, labels: 12,852
  라벨 없는 이미지: 0
  이미지 없는 라벨: 0
[validation] images: 14,472, labels: 14,472
  라벨 없는 이미지: 0
  이미지 없는 라벨: 0


## 6. dataset.yaml 생성

In [9]:
cfg = {
    'path': '/content',
    'train': 'images/train',
    'val':   'images/validation',
    'nc': 1,
    'names': ['pill'],
}
with open('/content/dataset.yaml', 'w') as f:
    yaml.dump(cfg, f, allow_unicode=True)
print(open('/content/dataset.yaml').read())

names:
- pill
nc: 1
path: /content
train: images/train
val: images/validation



## 1. 학습

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
results = model.train(
    data='/content/dataset.yaml',
    epochs=30,
    imgsz=640,
    batch=16,
    optimizer='AdamW',
    lr0=0.002,
    device=0,
    project='/content/runs',
    name='yolo11n_single_baseline',
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, i

## 2. 검증

In [ ]:
metrics = model.val()
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

## 3. 결과 GCS 저장

In [ ]:
import os
os.system(
    'gcloud storage cp -r /content/runs/yolo11n_single_baseline '
    'gs://pilliot-raw-data-2026/runs/yolo11n_single_baseline/'
)
print("GCS 저장 완료")